# Current integrated mushroom baselines

**Contribution owner: Muneeb Shakoor - integration, baseline reproduction, AutoML and error analysis (tasks 13, 15, 18, 24).**
Builds on Viviana Pajic's Mushroom exploration/evaluation (tasks 8/11) and the shared
AI-assisted starter maintained by Tomislav Novosel. Codex implemented and executed
this work at Muneeb's request on 8-9 October 2026. Muneeb selected the scope and
preservation requirements; this does not claim independent human authorship or
completed human code review. See [AI assistance](../docs/AI_USE.md) and
[reproduction and decisions](../docs/MUSHROOM_WORKFLOW.md).

Use Python 3.11 with `requirements-pycaret.lock.txt` for the recorded environment.


**CURRENT INTEGRATED WORKFLOW RESULT.** The original executed notebook is preserved
in [the historical snapshot](history/02_baseline_models_before_integration.ipynb).
The historical 60/20/20 metrics remain in `reports/mushroom_validation_metrics.csv`.
Their changed protocol prevents treating score differences as a measured improvement.


## Reproduce the agreed comparison

Use all 4,000 development rows with the same stratified five folds repeated twice
(seed 42). Every estimator contains its own fold-fitted preprocessing. The dummy
is a majority reference; LR tests an additive relationship; RF tests interactions.
These initial configurations are not a tuning exercise. Rank by mean fold average
precision, then examine ROC-AUC. Fold standard deviations are descriptive, not CIs.


In [1]:
from pathlib import Path
import sys
ROOT = Path.cwd()
while not (ROOT / 'src/mushrooms.py').exists():
    if ROOT == ROOT.parent:
        raise RuntimeError('Open the notebook inside the repository.')
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / 'src'))
import json
import pandas as pd
from mushroom_workflow import REPORTS, PROCESSED
from mushroom_workflow import run_baselines
results = run_baselines()
display(results[['model', 'cv_average_precision', 'cv_average_precision_std',
                 'cv_roc_auc', 'train_average_precision', 'mean_fit_seconds']].round(6))


Evaluating majority_baseline: 5 folds x 2 repeats...


  AP=0.378750, ROC-AUC=0.500000


Evaluating logistic_regression: 5 folds x 2 repeats...


  AP=0.630651, ROC-AUC=0.689445


Evaluating random_forest: 5 folds x 2 repeats...


  AP=0.783075, ROC-AUC=0.833420


Collision sensitivity: same RF, same validation rows; exclude fold-training matches.


              model  cv_average_precision  cv_roc_auc  oof_poison_recall
  majority_baseline              0.378750    0.500000           0.000000
logistic_regression              0.630651    0.689445           0.345875
      random_forest              0.783075    0.833420           0.520792


,model,cv_average_precision,cv_average_precision_std,cv_roc_auc,train_average_precision,mean_fit_seconds
0,majority_baseline,0.378750,0.000000,0.500000,0.378750,0.015551
1,logistic_regression,0.630651,0.010340,0.689445,0.653796,0.041113
2,random_forest,0.783075,0.008699,0.833420,0.950684,0.520128


## Separate ranking from operating decisions

The metrics below use threshold 0.5 on the mean of the two held-out probabilities
for each development row. They describe development behavior, not a reserved-test
result or guaranteed future recall. AP above is the mean of fold AP values, which
is a different summary from AP computed on pooled OOF predictions. The earlier
90% recall proposal is evaluated separately in notebook 04; it does not change these reference metrics.


In [2]:
display(results[['model', 'threshold', 'oof_accuracy', 'oof_poison_precision',
                 'oof_poison_recall', 'oof_f1', 'oof_tn', 'oof_fp', 'oof_fn', 'oof_tp']].round(4))


,model,threshold,oof_accuracy,oof_poison_precision,oof_poison_recall,oof_f1,oof_tn,oof_fp,oof_fn,oof_tp
0,majority_baseline,0.5,0.6212,0.0000,0.0000,0.0000,2485,0,1515,0
1,logistic_regression,0.5,0.7035,0.7288,0.3459,0.4691,2290,195,991,524
2,random_forest,0.5,0.7847,0.8539,0.5208,0.6470,2350,135,726,789


## Development-only collision sensitivity

Keep the same validation rows in every fold. For the sensitivity run only, exclude
training rows with exactly the same retained predictors as a validation row. This
is a conservative sensitivity check, not a canonical cleaning/deletion rule.
All other RF parameters and fold assignments remain fixed.


In [3]:
sensitivity = pd.read_csv(REPORTS / 'collision_sensitivity.csv')
reference = results.set_index('model').loc['random_forest']
display(sensitivity[['cv_average_precision', 'cv_roc_auc', 'collision_training_exclusions']])
print('Sensitivity AP minus reference AP:', sensitivity.iloc[0].cv_average_precision - reference.cv_average_precision)
print('Sensitivity ROC-AUC minus reference:', sensitivity.iloc[0].cv_roc_auc - reference.cv_roc_auc)


,cv_average_precision,cv_roc_auc,collision_training_exclusions
0,0.784446,0.834341,10


Sensitivity AP minus reference AP: 0.0013714855445569052
Sensitivity ROC-AUC minus reference: 0.0009203737275134127


## Interpretation and handoff

The saved run reproduces the earlier task-11 ranking: RF is strongest, with a large
training/development gap. The small collision sensitivity does not justify deleting
observations or changing the test membership. It also does not prove the source
observations are independent. This baseline checkpoint does not select a final model. Notebook 04 records the later provisional development threshold policy.

The runner exports one **provisional RF reference bundle** containing the complete
fitted pipeline, ten-input schema, class interpretation, threshold 0.5 and metadata.
The app consumes that bundle. It is a development demo, not a final deployed model.
Continue to notebook 03 for bounded AutoML screening.
